# Enterprise AI System Design — FastAPI Production Architecture

This is one of the most common topics in **EPAM, Microsoft, Deloitte, Accenture, Cognizant, and AWS** interviews.

Interviewers are not interested in whether you can write a simple FastAPI API—they want to know whether you can organize a **production-grade application**.

---

# 1. What is Production Architecture?

## Interview Answer

A production architecture organizes the application into multiple layers so that it is:

- Scalable
- Maintainable
- Testable
- Secure
- Reusable

Instead of writing everything in one file (`main.py`), responsibilities are separated.

---

# Bad Architecture

```text
main.py

- API
- JWT
- LangGraph
- Bedrock
- Qdrant
- SQL
- S3
- Logging

Everything in one file.
```

Problems

❌ Difficult to maintain

❌ Difficult to test

❌ Difficult to scale

---

# Production Architecture

```text
Client
   │
   ▼
API Layer
   │
   ▼
Service Layer
   │
   ▼
AI Layer
   │
   ▼
Repository Layer
   │
   ▼
Databases / AWS Services
```

Each layer has one responsibility.

---

# 2. Project Structure

## Azure + AWS (Same Structure)

```text
app/
│
├── main.py
│
├── api/
│   ├── chat.py
│   ├── auth.py
│   └── upload.py
│
├── services/
│   ├── chat_service.py
│   ├── auth_service.py
│   └── upload_service.py
│
├── agents/
│   └── langgraph_agent.py
│
├── rag/
│   ├── retriever.py
│   ├── embeddings.py
│   └── chunking.py
│
├── repositories/
│   ├── user_repository.py
│   └── chat_repository.py
│
├── database/
│   ├── postgres.py
│   └── redis.py
│
├── cloud/
│   ├── aws_bedrock.py
│   ├── s3.py
│   ├── secrets.py
│   ├── azure_openai.py
│   └── blob_storage.py
│
├── middleware/
│   ├── auth.py
│   ├── logging.py
│   └── exception.py
│
├── models/
│   ├── request.py
│   └── response.py
│
├── config/
│   └── settings.py
│
└── utils/
```

---

# 3. Layer Responsibilities

## API Layer

Responsible for

- Receive HTTP request
- Validate request
- Call service
- Return response

Never

❌ Call Bedrock

❌ Query SQL

❌ Business Logic

Example

```python
@router.post("/chat")
async def chat(request: ChatRequest):
    return await chat_service.ask(request)
```

---

## Service Layer

Responsible for

- Business Logic
- Validation
- Orchestration

Example

```text
Receive Question

↓

Call LangGraph

↓

Save Chat

↓

Return Response
```

Never

❌ SQL Queries

❌ HTTP Endpoints

---

## AI Layer

Responsible for

- LangGraph
- LangChain
- Prompt
- Tool Calling
- Memory
- Agent

Example

```text
Question

↓

Planner

↓

Retriever

↓

LLM

↓

Answer
```

---

## Repository Layer

Responsible for

Database communication.

Example

```python
save_chat()

get_user()

update_history()
```

Never

Business Logic.

---

## Cloud Layer

Responsible for

AWS

- Bedrock
- S3
- Secrets Manager

Azure

- Azure OpenAI
- Blob Storage
- Key Vault

No business logic here.

---

# 4. Request Flow

```text
User

↓

Route53 + CloudFront
Azure Front Door

↓

API Gateway
Azure API Management

↓

ALB
Azure Application Gateway

↓

FastAPI

↓

Chat API

↓

Chat Service

↓

LangGraph

↓

Retriever

↓

Qdrant / Azure AI Search

↓

AWS Bedrock / Azure OpenAI

↓

Answer

↓

Repository

↓

RDS PostgreSQL / Azure SQL

↓

JSON Response
```

---

# 5. Why Service Layer?

Suppose

Tomorrow

You add

WhatsApp

```text
REST API

↓

Service

↑

WhatsApp

↑

Slack

↑

Teams
```

All channels

Reuse

same

business logic.

---

# 6. Dependency Injection

FastAPI supports DI.

Example

```python
def get_db():
    return Session()

@app.get("/")
def home(db=Depends(get_db)):
    return {"status": "ok"}
```

Benefits

- Loose coupling
- Easier testing
- Reusable dependencies

---

# 7. Configuration Management

Never

```python
API_KEY="abc123"
```

Instead

AWS

Secrets Manager

Azure

Key Vault

Access through

```python
settings.OPENAI_KEY
```

---

# 8. Logging

Every request

↓

CloudWatch

Azure Monitor

Example

```python
logger.info("User logged in")
```

---

# 9. Exception Handling

Never

```python
raise Exception()
```

Use

Global Exception Handler

```python
@app.exception_handler(Exception)
```

Benefits

Consistent error responses.

---

# 10. Middleware

Responsibilities

- Logging
- JWT
- Request ID
- CORS
- Timing

Example

```python
@app.middleware("http")
async def log(request, call_next):
    response = await call_next(request)
    return response
```

---

# 11. Health Check

Required for

AWS ALB

Azure Application Gateway

Example

```python
@app.get("/health")
def health():
    return {"status": "healthy"}
```

Load Balancer calls

```text
GET /health
```

---

# 12. Async APIs

Instead of

```python
def chat():
```

Use

```python
async def chat():
```

Benefits

- Better concurrency
- Higher throughput
- Suitable for LLM calls

---

# 13. Background Tasks

Don't make the user wait.

Instead

Upload PDF

↓

Return

↓

Background

- OCR
- Chunk
- Embedding
- Qdrant

---

# 14. Streaming Response

Instead of waiting

15 seconds

Return

```text
Hello

Hello Suraj

Hello Suraj, your answer...
```

Used by

ChatGPT

Claude

Copilot

---

# 15. Authentication

Azure

Entra ID

JWT

AWS

Cognito

JWT

FastAPI

Validates token

↓

LangGraph

---

# 16. Production Deployment

AWS

```text
GitHub

↓

GitHub Actions

↓

Docker Build

↓

Amazon ECR

↓

Amazon ECS Fargate

↓

ALB

↓

FastAPI
```

Azure

```text
GitHub

↓

GitHub Actions

↓

Azure Container Registry

↓

Azure Container Apps

↓

Application Gateway

↓

FastAPI
```

---

# 17. Best Practices

✅ Layered architecture

✅ Stateless FastAPI

✅ JWT Authentication

✅ Async APIs

✅ Health endpoint

✅ Dependency Injection

✅ Global exception handling

✅ Configuration outside code

✅ Structured logging

✅ Dockerized deployment

---

# 18. Common Mistakes

❌ Everything in `main.py`

❌ SQL inside API

❌ LLM call inside route

❌ Hardcoded secrets

❌ No logging

❌ No health check

❌ No async

❌ No exception handling

❌ No dependency injection

---

# 19. Interview Questions

### Q1. Why layered architecture?

To separate responsibilities, improve maintainability, enable testing, and support scalability.

---

### Q2. Why Service Layer?

Business logic is reusable across multiple APIs and clients.

---

### Q3. Why Repository Layer?

To isolate database access from business logic.

---

### Q4. Why Dependency Injection?

Reduces coupling and makes components easier to test and replace.

---

### Q5. Why Async FastAPI?

LLM calls, database operations, and external APIs are I/O-bound. Async allows the server to handle other requests while waiting for those operations.

---

### Q6. Why Global Exception Handling?

Provides consistent error responses, centralized logging, and prevents leaking internal details.

---

# 20. Follow-up Questions

### Can FastAPI directly call Bedrock?

Yes.

Should it?

**No.**

Create a service:

```text
FastAPI

↓

Chat Service

↓

Bedrock Client
```

---

### Can LangGraph directly access PostgreSQL?

Technically yes.

Architecturally no.

Use

```text
LangGraph

↓

Service

↓

Repository

↓

Database
```

This keeps responsibilities separated.

---

### Should FastAPI store chat history?

No.

FastAPI orchestrates the request.

The repository layer persists chat history in:

- AWS → Amazon RDS PostgreSQL
- Azure → Azure SQL / PostgreSQL

---

# 21. EPAM Senior Answer (2–3 Minutes)

> "I follow a layered architecture for production FastAPI applications. The API layer is responsible only for request validation and routing. Business logic resides in the service layer, which orchestrates AI workflows using LangGraph. The AI layer handles prompt management, retrieval, tool calling, and interactions with AWS Bedrock or Azure OpenAI. Database access is isolated in the repository layer, while cloud-specific integrations such as Amazon S3, AWS Secrets Manager, Azure Blob Storage, or Azure Key Vault are encapsulated in dedicated cloud service classes. The application is fully asynchronous, secured with JWT authentication, uses dependency injection, structured logging, global exception handling, and health endpoints. It is containerized with Docker and deployed to Amazon ECS Fargate or Azure Container Apps behind a load balancer, with CloudWatch or Application Insights for monitoring and GitHub Actions for CI/CD. This architecture is modular, scalable, and suitable for enterprise AI applications."